# 🏥 Payer Claims Risk Engine v4 — Jupyter Notebook
### Complete Walkthrough: Train · Predict · AR Days Saved · Collection Projection · Dashboard

**New in v4:**
1. **AR Days Saved** — how many AR days each risk score and action eliminates  
2. **Success Rate + Collection Projection** — estimated recovery % and dollar amount per claim  
3. **No-Denial Handling** — engine reads prior activity notes when no denial code is present  
4. **Auto Dashboard** — self-contained HTML dashboard generated from output data  


## 1. Setup

In [ ]:
import sys, time, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')  # add repo root to path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8fafc',
                     'axes.grid':True,'grid.alpha':0.4,'font.size':11})

from python.payer_risk_engine import PayerRiskEngine

print("✅ Libraries loaded")


## 2. Load & Explore Training Data

In [ ]:
df_train = pd.read_csv("data/training_data.csv")
print(f"Training records: {len(df_train)}")
display(df_train)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

pc = df_train['Payer_Name'].value_counts()
axes[0].barh(pc.index, pc.values, color='#1d4ed8'); axes[0].set_title('Claims by Payer', fontweight='bold')

dc = df_train['Denial_Reason'].value_counts()
axes[1].bar(dc.index, dc.values, color=['#b91c1c','#b45309','#166534','#7c3aed','#0891b2'][:len(dc)])
axes[1].set_title('Denial Code Frequency', fontweight='bold')

axes[2].hist(df_train['Resolution_Days'], bins=8, color='#1d4ed8', alpha=0.8, edgecolor='white')
axes[2].axvline(100, color='#b45309', linestyle='--', label='Medium threshold')
axes[2].axvline(200, color='#b91c1c', linestyle='--', label='High threshold')
axes[2].set_title('Resolution Days', fontweight='bold'); axes[2].legend(fontsize=9)
plt.tight_layout(); plt.show()


## 3. Train the Engine

In [ ]:
engine = PayerRiskEngine(avg_claim_value=850.0)

t0 = time.time()
engine.train("data/training_data.csv")
print(f"Trained in {time.time()-t0:.3f}s | Records: {len(engine.training_df)}")


## 4. Single Claim Prediction (with all v4 outputs)

In [ ]:
# Claim WITH denial code
result = engine.predict_single(
    payer="Aetna", cpt="99213", dx="D44.90", denial="CO-50",
    notes="", amount=950.0
)
print("\n=== Claim WITH Denial Code ===")
for k, v in result.items():
    if not k.startswith('_'):
        print(f"  {k:<30} : {v}")


In [ ]:
# Claim WITHOUT denial code — uses prior notes (Feature 3)
result_nd = engine.predict_single(
    payer="Aetna", cpt="99213", dx="D44.90", denial="",
    notes="Added 25 modifier previously — claim pending review",
    amount=650.0
)
print("\n=== Claim WITHOUT Denial Code (No-Denial Handling) ===")
for k, v in result_nd.items():
    if not k.startswith('_'):
        print(f"  {k:<30} : {v}")


## 5. Batch Prediction

In [ ]:
df_input = pd.read_csv("data/input_claims.csv")
print(f"Input claims: {len(df_input)}")
display(df_input.head())


In [ ]:
t0 = time.time()
results = engine.predict("data/input_claims.csv")
elapsed = time.time() - t0

print(f"\n✅ {len(results):,} claims in {elapsed:.3f}s ({len(results)/elapsed:,.0f}/sec)")

show_cols = ['Payer_Name','CPT_Code','DX_Code','Denial_Reason','Predicted_Action',
             'Est_Resolution_Days','Risk_Level','Confidence','AR_Days_Saved',
             'AR_Pct_Saved','Est_Success_Rate','Est_Collection_Amount',
             'Action_Category','No_Denial_Claim']
display(results[[c for c in show_cols if c in results.columns]])


## 6. Summary Statistics

In [ ]:
s = engine.summary(results)
print("="*55)
print("  RISK ENGINE v4 — SUMMARY")
print("="*55)
for k, v in s.items():
    print(f"  {k:<25} : {v}")
print("="*55)


## 7. Results Visualisation

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Risk pie
rc = results['Risk_Level'].value_counts()
axes[0,0].pie(rc.values, labels=rc.index,
              colors=['#b91c1c','#b45309','#166534'][:len(rc)],
              autopct='%1.1f%%', startangle=90,
              wedgeprops={'edgecolor':'white','linewidth':2})
axes[0,0].set_title('Risk Level Distribution', fontweight='bold')

# Confidence bar
cc = results['Confidence'].value_counts()
axes[0,1].bar(cc.index, cc.values,
              color=['#166534','#b45309','#b91c1c'][:len(cc)],
              edgecolor='white', borderRadius=4 if False else 0)
axes[0,1].set_title('Confidence Level', fontweight='bold')

# AR Days Saved by category
if 'Action_Category' in results.columns:
    ar_by_cat = results.groupby('Action_Category')['AR_Days_Saved'].mean().sort_values(ascending=True)
    axes[0,2].barh(ar_by_cat.index, ar_by_cat.values, color='#1d4ed8', alpha=0.8)
    axes[0,2].set_title('Avg AR Days Saved by Category', fontweight='bold')
    axes[0,2].set_xlabel('Days Saved')

# Collection vs Writeoff
labels = ['Est. Collection', 'Est. Write-Off']
vals   = [s['total_collect'], s['total_writeoff']]
axes[1,0].bar(labels, vals, color=['#166534','#b91c1c'], edgecolor='white')
axes[1,0].set_title('Collection vs Write-Off', fontweight='bold')
axes[1,0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))

# Success rate by denial code
if 'Denial_Reason' in results.columns:
    sr = results[results['Denial_Reason']!='—'].groupby('Denial_Reason')['_success_rate_raw'].mean()*100
    colors_sr = ['#166534' if v>=70 else '#b45309' if v>=40 else '#b91c1c' for v in sr.values]
    axes[1,1].bar(sr.index, sr.values, color=colors_sr, edgecolor='white')
    axes[1,1].set_title('Est. Success Rate by Denial Code', fontweight='bold')
    axes[1,1].set_ylabel('%')
    axes[1,1].axhline(50, color='gray', linestyle='--', alpha=0.5)

# No-denial vs denial
nd_counts = results['No_Denial_Claim'].value_counts()
axes[1,2].pie(nd_counts.values, labels=['Has Denial' if k=='No' else 'No Denial Code' for k in nd_counts.index],
              colors=['#1d4ed8','#f59e0b'],
              autopct='%1.1f%%', startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
axes[1,2].set_title('No-Denial vs Denial Claims', fontweight='bold')

plt.suptitle('Payer Risk Engine v4 — Results Dashboard', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout(); plt.savefig('docs/dashboard_charts.png', dpi=150, bbox_inches='tight'); plt.show()
print("Saved: docs/dashboard_charts.png")


## 8. AR Days Saved Analysis (Feature 1)

In [ ]:
print("="*60)
print("  AR DAYS SAVED ANALYSIS")
print("="*60)
print(f"  Total AR days saved across all claims : {s['total_ar_saved']:,}")
print(f"  Average AR days saved per claim       : {s['avg_ar_saved']:.1f}")
print(f"  Average resolution days               : {s['avg_days']:.1f}")

if 'Action_Category' in results.columns:
    print("\n  AR Days Saved by Action Category:")
    ar_summary = results.groupby('Action_Category').agg(
        Claims=('AR_Days_Saved','count'),
        Avg_Saved=('AR_Days_Saved','mean'),
        Total_Saved=('AR_Days_Saved','sum'),
        Baseline=('AR_Baseline_Days','mean'),
    ).round(1).sort_values('Avg_Saved', ascending=False)
    display(ar_summary)
print("="*60)


## 9. Collection Projection Analysis (Feature 2)

In [ ]:
print("="*60)
print("  COLLECTION PROJECTION")
print("="*60)
print(f"  Total charge amount    : ${s['total_charge']:>12,.2f}")
print(f"  Est. collectible       : ${s['total_collect']:>12,.2f}")
print(f"  Est. write-off risk    : ${s['total_writeoff']:>12,.2f}")
print(f"  Overall success rate   : {s['success_rate']:>12}")

if 'Denial_Reason' in results.columns:
    print("\n  Collection Projection by Denial Code:")
    col_by_denial = results[results['Denial_Reason']!='—'].groupby('Denial_Reason').agg(
        Claims=('Charge_Amount','count'),
        Total_Charged=('Charge_Amount','sum'),
        Est_Collection=('Est_Collection_Amount','sum'),
        Est_WriteOff=('Est_Write_Off_Risk','sum'),
        Avg_Success=('_success_rate_raw','mean'),
    ).round(2)
    col_by_denial['Avg_Success'] = col_by_denial['Avg_Success'].apply(lambda x: f"{x*100:.1f}%")
    display(col_by_denial.sort_values('Total_Charged', ascending=False))
print("="*60)


## 10. No-Denial Claims (Feature 3)

In [ ]:
nd_claims = results[results['No_Denial_Claim']=='Yes']
print(f"No-denial claims: {len(nd_claims)}")

if len(nd_claims):
    show = ['Payer_Name','CPT_Code','DX_Code','Predicted_Action',
            'Est_Resolution_Days','Risk_Level','Confidence',
            'AR_Days_Saved','Est_Success_Rate']
    display(nd_claims[[c for c in show if c in nd_claims.columns]])


## 11. Export Results + Generate Dashboard (Feature 4)

In [ ]:
# Export full CSV
engine.export(results, "results/output_v4.csv")
print("✅ Full results: results/output_v4.csv")

# Export high-risk only
engine.export(results, "results/output_HIGH_RISK.csv", high_risk_only=True)
print(f"✅ High-risk only: results/output_HIGH_RISK.csv")

# Export Excel with color coding
engine.export(results, "results/output_v4.xlsx")
print("✅ Excel (color-coded): results/output_v4.xlsx")

# Generate HTML Dashboard
engine.generate_dashboard(results, "results/dashboard.html",
                          title="Payer Claims Risk Engine v4 — Live Dashboard")
print("✅ Dashboard: results/dashboard.html — open in any browser")


## 12. Bulk Performance Benchmark (2L+ Records)

In [ ]:
import random
random.seed(42); np.random.seed(42)

PAYERS  = ['Aetna','BCBS of SC','BCBS of CA','UHC','Medicare','Medicaid','Cigna','Humana','BCBS of TX']
CPTS    = ['99213','99214','99215','17000','17110','17111','18000','64633','64690']
DXS     = ['D44.90','D56.90','L82.10','SA41.00','M45.20','M54.50','M47.816','J45.20']
DENIALS = ['CO-50','CO-50','CO-50','CO-18','CO-30','CO-197','CO-4','CO-29','']

N = 50_000  # change to 200000 for full benchmark
synthetic = pd.DataFrame({
    'Payer_Name':   np.random.choice(PAYERS, N),
    'CPT_Code':     np.random.choice(CPTS, N),
    'DX_Code':      np.random.choice(DXS, N),
    'Denial_Reason':np.random.choice(DENIALS, N),
    'Notes':        ['Prior claim resubmitted with modifier' if d=='' else '' for d in np.random.choice(DENIALS, N)],
    'Charge_Amount':np.random.uniform(400, 5000, N).round(2),
})

print(f"Benchmarking {N:,} records...")
t0 = time.time()
bulk = engine.predict(synthetic)
elapsed = time.time() - t0

bs = engine.summary(bulk)
print(f"\n✅ {N:,} claims processed in {elapsed:.2f}s ({N/elapsed:,.0f} claims/sec)")
print(f"   Est. collection  : ${bs['total_collect']:,.0f}")
print(f"   Avg AR days saved: {bs['avg_ar_saved']:.1f}")
print(f"   Overall success  : {bs['success_rate']}")


---
## ✅ Notebook Complete

| Feature | Status |
|---|---|
| AR Days Saved calculation | ✅ Feature 1 |
| Success Rate + Collection Projection | ✅ Feature 2 |
| No-denial claim handling via notes | ✅ Feature 3 |
| Auto-generated HTML Dashboard | ✅ Feature 4 |

**All outputs in `/results/`:**  `output_v4.csv` · `output_HIGH_RISK.csv` · `output_v4.xlsx` · `dashboard.html`
